[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [PySpark & Parquet](PySpark-Parquet.md) | [Tasks](Tasks.md) | [Task 1](Task-1-aggregate-daily-traffic.md) | [Task 2](Task-2-daily-churn.md) | [Task 3](Task-3-batch-job.md) | [Task 4](Task-4-full-period-analysis.md) | Prototype Notebook ⮕ | [Analysis Notebook](analysis.ipynb)

# Iran (IR) traffic change prototype: 2026-02-10 to 2026-03-05

This notebook prototypes an analysis of UCSD Network Telescope FlowTuple traffic
attributed to Iran (IR) over the period **2026-02-10 to 2026-03-05**. The goal is:

1. Identify the source IP addresses and source ASNs geolocated to IR.
2. Aggregate the daily traffic volume (total packets, unique source IPs, unique
   ASNs) contributed by those IR-geolocated sources.

Running the full ~24-day period cell-by-cell in a single interactive Jupyter
kernel is slow and ties up a shared interactive allocation for a long time, so
this notebook only prototypes and sanity-checks the logic on **2 test days**
(2026-02-14 to 2026-02-15). Once the logic looks right here (Tasks 1 and 2),
you will package the same per-day code into `ir_traffic_batch.py` and submit it
for the full period via the Slurm job array in `run_ir_traffic.slurm` (Task 3 -
see the "Scaling up" section near the end of this notebook, and
[Tasks.md](Tasks.md)).

Replace every `# YOUR CODE HERE` section below with your own code, then answer
the questions in the markdown cells that follow each task.

In [ ]:
# Core Spark imports used throughout the notebook for session setup and DataFrame operations.
import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as pst

# Standard Python/data libraries.
import os
import socket
import struct
import pandas as pd
import matplotlib.pyplot as plt

# S3 client for object listing before Spark reads the Avro files.
import boto3
from botocore.config import Config

# Initialize Spark discovery so PySpark can find the local Spark installation paths.
import findspark
findspark.init()

### Setting environment variables here

In [ ]:
# These should be preloaded in your shell (.bashrc + .ucsdnts3.env), same as ft-demo.ipynb.
PROJECT = %env ETP26_ACCOUNT
USER=%env USER

UCSD_NT_S3_ACCESS_KEY = %env UCSD_NT_S3_ACCESS_KEY
UCSD_NT_S3_SECRET_KEY = %env UCSD_NT_S3_SECRET_KEY


PART1OUTPUTDIR = f'/expanse/lustre/projects/{PROJECT}/{USER}/output/prototype'

### Analysis configuration: target country and date ranges

In [ ]:
# Geolocation field + country code used to identify IR sources.
# netacq_country matches what the ft-demo.ipynb country-aggregation example uses;
# maxmind_country is kept as an easy alternative to cross-check against.
GEO_FIELD = 'netacq_country'
GEO_COUNTRY = 'IR'

# Full analysis period this study is ultimately about. This notebook does NOT loop
# over the whole thing - see the "Scaling up" section at the bottom for that.
FULL_START_DATE = '2026-02-10'
FULL_END_DATE = '2026-03-05'

# 2-day test window used to prototype and sanity-check the logic interactively.
TEST_DATES = ['2026-02-14','2026-02-15']

S3_ENDPOINT_URL = 'https://hermes.caida.org'
S3_BUCKET = 'telescope-ucsdnt-avro-flowtuple-v4-2026'

### List available FlowTuple files for the test days

In [ ]:
# Build an S3 client with explicit endpoint and credentials, same as ft-demo.ipynb.
s3 = boto3.client(
    's3',
    endpoint_url=S3_ENDPOINT_URL,
    aws_access_key_id=UCSD_NT_S3_ACCESS_KEY,
    aws_secret_access_key=UCSD_NT_S3_SECRET_KEY,
    config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
)

def list_avro_objects_for_date(date_str):
    """List FlowTuple objects for one YYYY-MM-DD day partition."""
    day = pd.to_datetime(date_str)
    prefix = f'datasource=ucsd-nt/year={day.year:04d}/month={day.month:02d}/day={day.day:02d}/'

    paginator = s3.get_paginator('list_objects')
    objects = []
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=prefix):
        objects.extend(page.get('Contents', []))

    df = pd.DataFrame(objects, columns=['Key', 'Size'])
    df['unix_time'] = df['Key'].str.extract(r'\.(\d+)\.flowtuple-v4\.avro$')[0].astype('int64')
    df['datetime'] = pd.to_datetime(df['unix_time'], unit='s', utc=True)
    return df

# Combined listing across all test days - the per-day loop below filters this by time range.
s3_objects_df = pd.concat([list_avro_objects_for_date(d) for d in TEST_DATES], ignore_index=True)
print(f'{len(s3_objects_df)} FlowTuple files found across the {len(TEST_DATES)} test days')
s3_objects_df.head()

### PySpark setup

In [ ]:
# Configure Spark to read Avro data from the CAIDA S3-compatible endpoint, same as ft-demo.ipynb.
# Arrow is disabled here because PySpark Arrow->pandas conversion can crash this runtime.
#
# spark.driver.memory matters a lot here: local[*] mode runs the driver and all "executors"
# in ONE JVM process, and Avro (unlike Parquet) has no column/predicate pushdown - Spark must
# fully decode every row of a whole day's ~288 files before the IR filter drops non-IR rows.
# Left unset, Spark defaults this to ~1g, so that JVM runs out of heap after a file or two.
# When that happens you see "SparkContext was shut down" / "Connection refused" - that's the
# whole JVM process dying (OS/cgroup OOM-kill), not an ordinary catchable Spark error, so the
# kernel needs a full RESTART afterwards (re-running cells reuses a dead py4j connection).
#
# Set this to comfortably below whatever --memory you gave `galyleo launch` (this notebook was
# launched with --memory 32, i.e. 32g TOTAL for the whole job) - leave several GB of headroom
# for the Python driver process, OS, and Jupyter/galyleo itself, not just the Spark JVM.

n_cores = os.environ.get('SLURM_CPUS_PER_TASK', '4')
memory = '24g'
resultsize = '4g'

conf = pyspark.SparkConf().setAll([
    ('spark.master', f'local[{n_cores}]'),
    ('spark.app.name', 'IR Traffic Prototype'),
    ('spark.jars.packages', 'org.apache.spark:spark-avro_2.13:4.2.0,org.apache.hadoop:hadoop-aws:3.5.0'),
    ('spark.hadoop.fs.s3a.endpoint', S3_ENDPOINT_URL),
    ('spark.hadoop.fs.s3a.path.style.access', 'true'),
    ('spark.hadoop.fs.s3a.endpoint.region', 'us-east-1'),
    ('spark.hadoop.fs.s3a.access.key', UCSD_NT_S3_ACCESS_KEY),
    ('spark.hadoop.fs.s3a.secret.key', UCSD_NT_S3_SECRET_KEY),
    ('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider'),
    # hadoop-aws 3.5.0 turns on the new S3 Analytics Accelerator reader by default; its
    # background prefetch threads can throw "SocketException: Socket closed" ERROR-level
    # noise when spark.stop() tears down the S3 connection pool out from under them at the
    # end of the session. Harmless (fires after outputs are already written), but disable it
    # to avoid the noise and this still-experimental component's other rough edges.
    ('spark.hadoop.fs.s3a.analytics.accelerator.enabled', 'false'),
    ('spark.sql.execution.arrow.pyspark.enabled', 'false'),
    ('spark.ui.showConsoleProgress', 'true') ,
    ('spark.driver.memory', memory),          # <-- match to your actual Jupyter --memory allocation
    ('spark.driver.maxResultSize', resultsize),
    ('spark.sql.shuffle.partitions', '100'),  # default 200 is excessive overhead for 4 local cores
])

spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN") 

### Handy IP helper function

In [ ]:
# Convert integer IPv4 (as stored in the src_ip column) into a dotted-quad string.
def long2ip(long):
    return socket.inet_ntoa(struct.pack('!L', long))

### Functions to facilitate loading FlowTuple files for a given day

In [ ]:
# Read one or more FlowTuple Avro files into Spark and normalize time to UTC timestamp.
def load_ft_pyspark(avro_files):
    df = spark.read.format('avro').load(avro_files)
    df = df.withColumn('time', F.to_utc_timestamp((df.time).cast(dataType=pst.TimestampType()), 'UTC'))
    return df

# Return S3A paths for objects whose extracted timestamps are within [start_time, end_time].
def time_to_avro_list(s3_objects_df, start_time, end_time):
    start_time = pd.Timestamp(start_time)
    end_time = pd.Timestamp(end_time)

    if start_time.tzinfo is None:
        start_time = start_time.tz_localize('UTC')
    if end_time.tzinfo is None:
        end_time = end_time.tz_localize('UTC')

    mask = (s3_objects_df['datetime'] >= start_time) & (s3_objects_df['datetime'] <= end_time)
    keys = s3_objects_df.loc[mask, 'Key'].tolist()
    return [f's3a://{S3_BUCKET}/{key}' for key in keys]

## Task 1: Identify IR-geolocated sources and aggregate daily traffic (prototype loop over test days)

For each test day: load the full day of FlowTuple files once, keep only rows
whose `netacq_country` is `IR`, then compute all four outputs we care about
from that same filtered DataFrame:
 - the daily totals: packet volume, unique source IPs, unique ASNs
 - the distinct `(src_ip, prefix2asn)` pairs geolocated to IR that day
 - the total packets contributed by each ASN that day
 - per-hour totals: packet volume and unique ASNs, for an hourly traffic trend

See the [Task 1 Guide](Task-1-aggregate-daily-traffic.md) for the Spark patterns
you'll need for each of these four outputs.

In [ ]:
import gc

daily_summaries = []
ir_src_asn_frames = []
asn_daily_frames = []
hourly_frames = []

for date_str in TEST_DATES:
    # For each date in TEST_DATES:
    # 1. Convert the date string to a pandas Timestamp and define the start and end of the day.
    # 2. Use time_to_avro_list to get the list of Avro files for that day.
    # 3. Load the Avro files into a Spark DataFrame using load_ft_pyspark.
    # 4. Filter the DataFrame for rows where the GEO_FIELD matches GEO_COUNTRY.
    # 5. Aggregate the filtered DataFrame to compute total packet count, unique source IPs, and unique ASNs, and append the results to daily_summaries.
    # 6. Collect distinct source IP and ASN pairs, convert to a pandas DataFrame, and append to ir_src_asn_frames.
    # 7. Group by ASN to compute total packet counts per ASN, convert to a pandas DataFrame, and append to asn_daily_frames.
    # 8. Group by hour to compute total packet counts and unique ASN counts per hour, convert to a pandas DataFrame, and append to hourly_frames.
    # 9. Clear Spark cache and delete intermediate DataFrames to manage memory usage before the next iteration.

    # YOUR CODE HERE

    # Release this day's DataFrames before the next iteration loads a fresh one - Spark
    # accumulates cached query plans/broadcast state across iterations in local mode, and
    # this keeps that from compounding into an OOM over a long loop.
    del spark_df, ir_df, src_asn_rows, asn_rows, hourly_rows
    spark.catalog.clearCache()
    gc.collect()

daily_summary_df = pd.DataFrame(daily_summaries)
ir_src_asn_df = pd.concat(ir_src_asn_frames, ignore_index=True)
asn_daily_packet_df = pd.concat(asn_daily_frames, ignore_index=True)
asn_daily_packet_df = asn_daily_packet_df.sort_values(
    ['date', 'total_packet_cnt'], ascending=[True, False]
).reset_index(drop=True)
hourly_packet_df = pd.concat(hourly_frames, ignore_index=True).sort_values('hour').reset_index(drop=True)

daily_summary_df

**Q1:** Compare the daily summary (total packets, unique source IPs, unique ASNs)
between the two test days. How much do they vary day to day, and what does that
suggest about how stable this baseline "background" telescope traffic is?

_Your answer here._

In [ ]:
os.makedirs(PART1OUTPUTDIR, exist_ok=True)

asn_daily_packet_df.to_parquet(os.path.join(PART1OUTPUTDIR, 'ir_asn_daily_packets_test.parquet'), index=False)
print(f'Wrote {os.path.join(PART1OUTPUTDIR, "ir_asn_daily_packets_test.parquet")}')

### Preview: distinct IR-geolocated source IPs / ASNs across the test window

In [ ]:
print(f"{ir_src_asn_df['src_ip'].nunique()} unique IR source IPs, "
      f"{ir_src_asn_df['prefix2asn'].nunique()} unique IR ASNs across {TEST_DATES}")
ir_src_asn_df.sort_values(by=['date', 'prefix2asn']).head(20)

## Task 2: Day-over-day churn at /24 granularity: (subnet, ASN) pairs that disappear

First aggregate each day's IR-geolocated source IPs to their containing /24
subnet (paired with the ASN seen for that subnet). Then diff the sets of
`(subnet_24, prefix2asn)` pairs between consecutive days to find the ones
present on the earlier day but no longer observed on the following day.

See the [Task 2 Guide](Task-2-daily-churn.md) for the set-difference pattern
you'll need.

In [ ]:
def ip_to_slash24(ip):
    return '.'.join(ip.split('.')[:3]) + '.0/24'

sorted_test_dates = sorted(TEST_DATES)

# Aggregate each day's IR-geolocated source IPs to distinct (subnet_24, prefix2asn) pairs.
ir_src_asn_df['subnet_24'] = ir_src_asn_df['src_ip'].apply(ip_to_slash24)
ir_subnet24_asn_df = ir_src_asn_df[['date', 'subnet_24', 'prefix2asn']].drop_duplicates()

disappeared_subnet24_frames = []

for prev_date, next_date in zip(sorted_test_dates[:-1], sorted_test_dates[1:]):
    # For each consecutive pair of dates in sorted_test_dates:
    # 1. Filter ir_subnet24_asn_df to get the set of (subnet_24, prefix2asn) pairs for prev_date and next_date.
    # 2. Compute the set difference to find pairs present on prev_date but not on next_date.
    # 3. Print how many pairs disappeared, and append a DataFrame (with prev_date/next_date columns) to disappeared_subnet24_frames.

    # YOUR CODE HERE

disappeared_subnet24_df = pd.concat(disappeared_subnet24_frames, ignore_index=True)
disappeared_subnet24_df = disappeared_subnet24_df[['prev_date', 'next_date', 'subnet_24', 'prefix2asn']]

disappeared_subnet24_df

**Q2:** Why look at churn in terms of `/24` subnets rather than individual source
IPs or whole ASNs? What real-world explanations (besides a country-wide outage)
could cause a `/24` to "disappear" for a day?

_Your answer here._

In [ ]:
os.makedirs(PART1OUTPUTDIR, exist_ok=True)

disappeared_subnet24_df.to_parquet(os.path.join(PART1OUTPUTDIR, 'ir_disappeared_subnet24_test.parquet'), index=False)
print(f'Wrote {os.path.join(PART1OUTPUTDIR, "ir_disappeared_subnet24_test.parquet")}')

### Save hourly traffic trend (test window) to Parquet

`hourly_packet_df` (total packets + unique ASNs per hour) only lives in this
kernel's memory so far. Persist it to Parquet here so it can be loaded and
plotted from `analysis.ipynb` without needing a live Spark session.

In [ ]:
os.makedirs(PART1OUTPUTDIR, exist_ok=True)

hourly_packet_df.to_parquet(os.path.join(PART1OUTPUTDIR, 'ir_hourly_packets_test.parquet'), index=False)
print(f'Wrote {os.path.join(PART1OUTPUTDIR, "ir_hourly_packets_test.parquet")}')

### Save prototype (test-window) outputs to Parquet

In [ ]:
os.makedirs(PART1OUTPUTDIR, exist_ok=True)

daily_summary_df.to_parquet(os.path.join(PART1OUTPUTDIR, 'ir_daily_summary_test.parquet'), index=False)
ir_src_asn_df.to_parquet(os.path.join(PART1OUTPUTDIR, 'ir_src_asn_test.parquet'), index=False)

print(f'Wrote {os.path.join(PART1OUTPUTDIR, "ir_daily_summary_test.parquet")}')
print(f'Wrote {os.path.join(PART1OUTPUTDIR, "ir_src_asn_test.parquet")}')

### Task 3: Scaling up - running the full 2026-02-10 to 2026-03-05 period as a Slurm batch job

The loop above works, but running all ~24 days of full-day Spark loads inside one
interactive kernel is slow and holds a shared interactive allocation the whole time.
Adapt the scripts and image from the [tutorial](https://github.com/CAIDA/nids-expanse-2026/tree/main/3_running_batch_job),
and complete `ir_traffic_batch.py` (in this same directory) to port Task 1's logic
into a standalone script, then run it as a Slurm job array covering the data from
Feb 10 - Mar 5, 2026. Also, sample one hour (00:00 UTC) per day for this analysis.
See the [Task 3 Guide](Task-3-batch-job.md) for the prototype-to-batch porting
pattern, including where it intentionally differs from Task 1.

`run_ir_traffic.slurm` is already provided - once `ir_traffic_batch.py` is complete
and you've added your email to the mail-notification line:

```bash
sbatch run_ir_traffic.slurm
squeue -u <username>          # check progress
```

Logs land in `joblogs/slurm-<array_job_id>_<array_task_id>.out`. Once finished,
the 24 sets of daily Parquet files are under
`/expanse/lustre/projects/${ETP26_ACCOUNT}/<username>/output/ir_traffic/<array_job_id>/`.

See [Tasks.md](Tasks.md) for Task 3's question, and continue to
[analysis.ipynb](analysis.ipynb) for Task 4 once the batch job completes.

**Q3:** Use your email notification to estimate the job run time. Inspect the configuration of the slurm script. It used array of jobs (one per day). How does the array of jobs help speed up the processing comparing to the Jupyter Notebook approach?

_Your answer here._

In [ ]:
spark.stop()

### End of prototype notebook.

[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [PySpark & Parquet](PySpark-Parquet.md) | [Tasks](Tasks.md) | [Task 1](Task-1-aggregate-daily-traffic.md) | [Task 2](Task-2-daily-churn.md) | [Task 3](Task-3-batch-job.md) | [Task 4](Task-4-full-period-analysis.md) | Prototype Notebook ⮕ | [Analysis Notebook](analysis.ipynb)